In [1]:
import warnings
warnings.filterwarnings("ignore")

In [2]:
%%capture
!pip install -q llama-index llama-index-llms-huggingface llama-index-embeddings-huggingface sentence-transformers transformers accelerate torch llama-index-readers-file pymupdf --disable-pip-version-check --root-user-action=ignore

### Environment Setup and Hugging Face Login

In [3]:
import torch # Base PyTorch library, Gemma runs on this
from transformers import AutoModelForCausalLM, AutoTokenizer # For loading Gemma from Hugging Face

# LlamaIndex specific imports
from llama_index.core import Settings # This is a central place to configure defaults in LlamaIndex
from llama_index.llms.huggingface import HuggingFaceLLM # A wrapper to make our Hugging Face Gemma model compatible with LlamaIndex
from llama_index.embeddings.huggingface import HuggingFaceEmbedding # A wrapper for using Hugging Face embedding models

In [4]:
from huggingface_hub import notebook_login
notebook_login()

### Load and Configure Gemma LLM and Embedding Model

In [5]:
# Load Gemma 2 2B model and tokenizer
model_id_gemma = "google/gemma-2-2b-it"
tokenizer_gemma = AutoTokenizer.from_pretrained(model_id_gemma)
model_gemma = AutoModelForCausalLM.from_pretrained(
    model_id_gemma,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)
print(f"\nGemma 2 2B loaded")

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/4.24M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/838 [00:00<?, ?B/s]

...

model-00001-of-00002.safetensors:   0%|          | 0.00/4.99G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/241M [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/187 [00:00<?, ?B/s]


Gemma 2 2B loaded


In [6]:
# System prompt for Gemma to ensure context-based answers
system_prompt = "You are a helpful and precise Q&A assistant. Your primary goal is to answer questions based *only* on the contextual information provided to you. If the answer cannot be found within the provided context, please state clearly 'The provided context does not contain the answer to this question.'"

# Create LlamaIndex SLM wrapper for Gemma
llm_gemma = HuggingFaceLLM(
    model=model_gemma,
    tokenizer=tokenizer_gemma,
    context_window=8192,
    max_new_tokens=500,
    generate_kwargs={"temperature": 0.2, "do_sample": True},
    system_prompt=system_prompt,
    device_map="auto"
)

# Set Gemma as default LLM for LlamaIndex
Settings.llm = llm_gemma
print(f"\nGemma 2 2B SLM wrapped and set as default in LlamaIndex Settings.")


Gemma 2 2B SLM wrapped and set as default in LlamaIndex Settings.


In [7]:
# Hugging Face embedding model ID
embed_model_id = "sentence-transformers/all-MiniLM-L6-v2"

# Initialize LlamaIndex embedding model
embed_model = HuggingFaceEmbedding(model_name=embed_model_id)

# Set default embedding model for LlamaIndex
Settings.embed_model = embed_model

# Set default chunk size for document splitting
Settings.chunk_size = 512

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

### Load Document and Build Vector Index

### Building the Q&A Pipeline

In [ ]:
from llama_index.core import SimpleDirectoryReader, Document

pdf_filename = "Data/Artificial intelligence - Wikipedia.pdf"

print(f"\nAttempting to load the document: {pdf_filename}...")
try:
    loader = SimpleDirectoryReader(input_files=[pdf_filename])
    documents = loader.load_data()

    if documents:
        print(f"Successfully loaded {len(documents)} 'Document' object(s) from the PDF.")
    else:
        print("No documents were loaded. Please double-check the filename and ensure it was uploaded correctly.")

except FileNotFoundError:
    print(f"ERROR: The file '{pdf_filename}' was not found in your Colab session storage.")
    print("Please make sure you've uploaded it and the filename matches exactly.")

except Exception as e:
    print(f"An unexpected error occurred while loading the PDF: {e}")


Attempting to load the document: /content/Data/Artificial intelligence - Wikipedia.pdf...
Successfully loaded 57 'Document' object(s) from the PDF.


In [9]:
from llama_index.core import VectorStoreIndex

print(f"\nBuilding the VectorStoreIndex from the loaded document(s)...")
# Process documents and build a searchable vector index
index = VectorStoreIndex.from_documents(documents)
print(f"Index built successfully!")


Building the VectorStoreIndex from the loaded document(s)...
Index built successfully!


In [10]:
query_engine = index.as_query_engine()

### Query Engine Demonstration

In [11]:
question1 = "What are some of the techniques used in AI research?"
print(f"Question 1: {question1}")

# Send the question to the query engine
response1 = query_engine.query(question1)

print(f"\nAnswer 1: {response1}")

Question 1: What are some of the techniques used in AI research?

Answer 1: 
Some of the techniques used in AI research include state space search, local search, heuristics, and adversarial search. 



In [12]:
question2 = "What is the difference between AI and AGI?"
print(f"Question 2: {question2}")
response2 = query_engine.query(question2)
print(f"\nAnswer 2: {response2}")

Question 2: What is the difference between AI and AGI?

Answer 2: 
The provided context states: "Some companies, such as OpenAI, Google DeepMind and Meta, aim to create artificial general intelligence (AGI) – AI that can complete virtually any cognitive task at least as well as a human." 
Therefore, the difference between AI and AGI is that AGI is a type of AI that can complete virtually any cognitive task at least as well as a human. 



In [13]:
question3 = "Can you explain NLP to me like I'm 5? Make it detailed"
print(f"Question 3: {question3}")
response3 = query_engine.query(question3)
print(f"\nAnswer 3: {response3}")

Question 3: Can you explain NLP to me like I'm 5? Make it detailed

Answer 3: 
Imagine you have a robot friend who loves to talk and understand you. You want to teach your robot friend how to talk and understand human language. 

NLP is like giving your robot friend special tools and instructions to help them learn how to talk and understand human language. 

These tools and instructions are like magic spells that help the robot understand words, sentences, and even whole conversations. 

For example, if you say "The cat is sleeping," the robot can understand what you mean because it has learned how to recognize words and their meanings. 

NLP is like giving your robot friend a special dictionary that helps them understand the different meanings of words. 

It's like teaching your robot friend how to read and write, but for human language. 

NLP helps robots understand and use human language in many ways, like talking to people, writing emails, and even playing games. 





### Chat Engine Demonstration

In [14]:
# Create a chat engine suitable for conversation
chat_engine = index.as_chat_engine(
    chat_mode="condense_question", # Condense question with chat history
    verbose=True # Enable verbose output for debugging
)
print(f"Chat engine created successfully! It's ready for a conversation.")

Chat engine created successfully! It's ready for a conversation.


In [15]:
# First question for the chat engine
q1 = "Please list six current, real-world applications of artificial intelligence, numbering them 1-6."
print(f"User: {q1}")
r1 = chat_engine.chat(q1)
print(f"Assistant: {r1}")

User: Please list six current, real-world applications of artificial intelligence, numbering them 1-6.
Querying with: Please list six current, real-world applications of artificial intelligence, numbering them 1-6.
Assistant: 
1. Advanced web search engines
2. Chatbots
3. Virtual assistants
4. Autonomous vehicles
5. Play and analysis in strategy games (e.g., chess and Go)
6. Generative AI (image, audio, and video generation from text prompts) 



In [16]:
# Follow-up question for the chat engine
q2 = "Can you elaborate a little more on 5?"
print(f"User: {q2}")
r2 = chat_engine.chat(q2)
print(f"Assistant: {r2}")

User: Can you elaborate a little more on 5?
Querying with: What are some real-world applications of generative AI, specifically in the context of strategy games like chess and Go? 


**Explanation:**

The standalone question captures the following:

* **Generative AI:**  The user is specifically asking about applications of generative AI.
* **Strategy Games:** The user is interested in applications within the context of strategy games like chess and Go. 
* **Real-World Applications:** The user wants to know about applications that are currently used in the real world. 


Let me know if you'd like to try another example! 

Assistant: 
The provided text does not contain the answer to this question. 



In [17]:
# Clear conversational history
chat_engine.reset()